In [1]:
import pandas as pd

facilities = pd.read_csv("/Users/othree/AIsummit/data/facilities.csv")
pincode = pd.read_csv("/Users/othree/AIsummit/data/pincode.csv")
nfhs = pd.read_csv("/Users/othree/AIsummit/data/nfhs.csv")

merge = pd.read_csv("/Users/othree/AIsummit/data/merge.csv")

v3 = pd.read_csv("/Users/othree/AIsummit/data/v3.csv")

In [2]:
for name, df in [('facilities', facilities), ('pincode', pincode), ('nfhs', nfhs)]:
    print(f"{'='*50}")
    print(f"  {name}  —  {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"{'='*50}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    summary = pd.DataFrame({
        'dtype': df.dtypes,
        'missing': missing,
        'missing_%': missing_pct,
    })
    print(summary.to_string())
    print()

  facilities  —  10088 rows × 51 columns
                                                   dtype  missing  missing_%
unique_id                                         object        0        0.0
source_types                                      object       78        0.8
source_ids                                        object       87        0.9
source_content_id                                 object       69        0.7
name                                              object       65        0.6
organization_type                                 object       61        0.6
content_table_id                                  object       62        0.6
phone_numbers                                     object      340        3.4
officialPhone                                     object      564        5.6
email                                             object     1518       15.0
websites                                          object       61        0.6
officialWebsite                    

## Step 1. Clean facilities.address_zipOrPostcode

In [3]:
# Strip spaces and accept only 6-digit numbers as valid pincodes; others become NaN
facilities['pincode_clean'] = (
    facilities['address_zipOrPostcode']
    .astype(str)
    .str.replace(' ', '', regex=False)
    .str.strip()
)
# Set values not matching 6-digit pattern to NaN
facilities.loc[~facilities['pincode_clean'].str.match(r'^\d{6}$'), 'pincode_clean'] = None

before = facilities['address_zipOrPostcode'].notna().sum()
after = facilities['pincode_clean'].notna().sum()
print(f"Before cleaning non-null: {before}")
print(f"After cleaning valid pincode: {after}")
print(f"Removed: {before - after}")
print()
print("Sample:")
print(facilities[['address_zipOrPostcode', 'pincode_clean']].head(10).to_string())

Before cleaning non-null: 10022
After cleaning valid pincode: 9926
Removed: 96

Sample:
  address_zipOrPostcode pincode_clean
0                110002        110002
1                122002        122002
2                700027        700027
3                400078        400078
4                209217        209217
5                422002        422002
6                562114        562114
7                122001        122001
8               201 301        201301
9                695009        695009


## Step 2. Unify pincode.pincode dtype

In [4]:
# Unify to string (same type as facilities' pincode_clean)
pincode['pincode_clean'] = pincode['pincode'].astype(str).str.strip()

print(f"pincode dtype converted: int64 → {pincode['pincode_clean'].dtype}")
print(f"Sample: {pincode['pincode_clean'].head(5).tolist()}")

# Check match rate
fac_pins = facilities['pincode_clean'].dropna()
matched = fac_pins.isin(pincode['pincode_clean']).sum()
print(f"\nMatch rate: {matched} / {len(fac_pins)} ({matched/len(fac_pins)*100:.1f}%)")

pincode dtype converted: int64 → object
Sample: ['504273', '504299', '504299', '504296', '504296']

Match rate: 9714 / 9926 (97.9%)


## Step 3. Normalize state names (pincode ↔ nfhs)

In [5]:
def normalize_state(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = s.replace('&', 'and')
    # Remove leading 'the '
    if s.startswith('the '):
        s = s[4:]
    return s

pincode['state_clean'] = pincode['statename'].apply(normalize_state)
nfhs['state_clean'] = nfhs['state_ut'].apply(normalize_state)

pin_states = set(pincode['state_clean'].dropna().unique())
nfhs_states = set(nfhs['state_clean'].dropna().unique())

print(f"pincode unique states: {len(pin_states)}")
print(f"nfhs unique states: {len(nfhs_states)}")
print(f"Matched: {len(pin_states & nfhs_states)}")
print()
print("States only in pincode:")
print(sorted(pin_states - nfhs_states))
print()
print("States only in nfhs:")
print(sorted(nfhs_states - pin_states))

pincode unique states: 36
nfhs unique states: 36
Matched: 34

States only in pincode:
['delhi', 'maharashtra']

States only in nfhs:
['maharastra', 'nct of delhi']


## Step 4. Normalize district names (pincode ↔ nfhs)

In [6]:
def normalize_district(s):
    if pd.isna(s):
        return None
    return str(s).strip().lower()

pincode['district_clean'] = pincode['district'].apply(normalize_district)
nfhs['district_clean'] = nfhs['district_name'].apply(normalize_district)

pin_districts = set(pincode['district_clean'].dropna().unique())
nfhs_districts = set(nfhs['district_clean'].dropna().unique())

exact_match = len(pin_districts & nfhs_districts)
print(f"pincode unique districts: {len(pin_districts)}")
print(f"nfhs unique districts: {len(nfhs_districts)}")
print(f"exact match: {exact_match}")
print(f"Match rate (nfhs basis): {exact_match}/{len(nfhs_districts)} ({exact_match/len(nfhs_districts)*100:.1f}%)")
print()
print("Districts only in nfhs (sample 20) — fuzzy matching needed:")
print(sorted(nfhs_districts - pin_districts)[:20])

pincode unique districts: 749
nfhs unique districts: 698
exact match: 597
Match rate (nfhs basis): 597/698 (85.5%)

Districts only in nfhs (sample 20) — fuzzy matching needed:
['ahmadnagar', 'allahabad', 'aravali', 'badgam', 'bandipore', 'bangalore', 'bangalore rural', 'bara banki', 'baramula', 'baudh', 'belgaum', 'bellary', 'bid', 'buldana', 'chamarajanagar', 'charkhi dadri', 'chhota udaipur', 'chikmagalur', 'chittaurgarh', 'dadra & nagar haveli']


## Step 5. Manual state mapping (resolve 2 mismatches)

In [7]:
# Align nfhs state_clean to pincode naming
# maharastra → maharashtra, nct of delhi → delhi
state_fix = {
    'maharastra': 'maharashtra',
    'nct of delhi': 'delhi',
}
nfhs['state_clean'] = nfhs['state_clean'].replace(state_fix)

# Verify
pin_states = set(pincode['state_clean'].dropna().unique())
nfhs_states = set(nfhs['state_clean'].dropna().unique())
print(f"Matched: {len(pin_states & nfhs_states)} / {len(nfhs_states)}")
print("Remaining mismatches:", sorted((pin_states ^ nfhs_states)))

Matched: 36 / 36
Remaining mismatches: []


## Step 6. District fuzzy matching (resolve 101 unmatched)

In [8]:
from difflib import get_close_matches

# Fuzzy match within the same state only (prevent cross-state false matches)
# Build (state_clean, district_clean) pairs from pincode
pin_by_state = pincode.dropna(subset=['state_clean', 'district_clean']) \
    .groupby('state_clean')['district_clean'].apply(set).to_dict()

# Only process rows not already exact-matched
nfhs['district_fuzzy'] = nfhs['district_clean']  # default: keep original value

fuzzy_map = {}
no_match = []

for _, row in nfhs.iterrows():
    state = row['state_clean']
    district = row['district_clean']
    # Skip if already exact matched
    pin_districts_in_state = pin_by_state.get(state, set())
    if district in pin_districts_in_state:
        continue
    # Fuzzy match only within the same state
    matches = get_close_matches(district, pin_districts_in_state, n=1, cutoff=0.80)
    if matches:
        fuzzy_map[district] = matches[0]
    else:
        no_match.append((state, district))

nfhs['district_fuzzy'] = nfhs['district_clean'].replace(fuzzy_map)

print(f"Fuzzy matches found (within-state): {len(fuzzy_map)}")
print(f"Still unmatched: {len(no_match)}")
print()
print("[Mapping results]")
for k, v in sorted(fuzzy_map.items()):
    print(f"  '{k}' → '{v}'")
print()
print("[Unmatched (state, district)]")
for s, d in sorted(no_match):
    print(f"  {s} / {d}")

Fuzzy matches found (within-state): 60
Still unmatched: 44

[Mapping results]
  'ahmadnagar' → 'ahmednagar'
  'aravali' → 'arvalli'
  'badgam' → 'budgam'
  'bandipore' → 'bandipora'
  'bangalore rural' → 'bengaluru rural'
  'bara banki' → 'barabanki'
  'baramula' → 'baramulla'
  'baudh' → 'boudh'
  'buldana' → 'buldhana'
  'chamarajanagar' → 'chamarajanagara'
  'charkhi dadri' → 'charki dadri'
  'chhota udaipur' → 'chhotaudepur'
  'chikmagalur' → 'chikkamagaluru'
  'chittaurgarh' → 'chittorgarh'
  'dadra & nagar haveli' → 'dadra and nagar haveli'
  'darjiling' → 'darjeeling'
  'davanagere' → 'davangere'
  'debagarh' → 'deogarh'
  'dhaulpur' → 'dholpur'
  'east jantia hills' → 'east jaintia hills'
  'faizabad' → 'firozabad'
  'firozpur' → 'firozepur'
  'gondiya' → 'gondia'
  'hardwar' → 'haridwar'
  'jalor' → 'jalore'
  'janjgir - champa' → 'janjgir-champa'
  'jhunjhunun' → 'jhunjhunu'
  'kabeerdham' → 'kabirdham'
  'kancheepuram' → 'kanchipuram'
  'kodagaon' → 'kondagaon'
  'kodarma' →

## Step 7. Manual district mapping (renamed places + fix wrong fuzzy matches)

In [9]:
# Before manual mapping — check what names pincode uses for states with unmatched districts
check_cases = {
    'uttar pradesh': ['allahabad', 'jyotiba phule nagar', 'kanshiram nagar', 'mahamaya nagar'],
    'karnataka': ['bangalore', 'belgaum', 'bellary', 'gulbarga', 'mysore', 'bijapur'],
    'haryana': ['gurgaon', 'mewat'],
    'west bengal': ['haora', 'hugli', 'koch bihar', 'dakshin dinajpur', 'uttar dinajpur',
                    'north twenty four pargana', 'south twenty four pargana', 'paschim medinipur', 'purba medinipur'],
    'jharkhand': ['pashchimi singhbhum', 'purbi singhbhum'],
    'uttarakhand': ['garhwal'],
    'punjab': ['muktsar', 'sahibzada ajit singh nagar'],
    'maharashtra': ['bid', 'raigarh'],
    'chhattisgarh': ['koriya', 'uttar bastar kanker'],
    'assam': ['kamrup metropolitan'],
    'gujarat': ['the dangs'],
    'andhra pradesh': ['sri potti sriramulu nello'],
    'telangana': ['warangal rural', 'warangal urban'],
    'tamil nadu': ['thoothukkudi'],
    'odisha': ['subarnapur'],
    'jammu and kashmir': ['punch'],
    'madhya pradesh': ['khandwa (east nimar)', 'khargone (west nimar)'],
    'puducherry': ['puducherry'],
    'mizoram': ['chandel'],
}

for state, districts in check_cases.items():
    pin_d = pin_by_state.get(state, set())
    print(f"\n[{state}] pincode districts:")
    print(sorted(pin_d))


[uttar pradesh] pincode districts:
['agra', 'aligarh', 'ambedkar nagar', 'amethi', 'amroha', 'auraiya', 'ayodhya', 'azamgarh', 'baghpat', 'bahraich', 'ballia', 'balrampur', 'banda', 'barabanki', 'bareilly', 'basti', 'bhadohi', 'bijnor', 'budaun', 'bulandshahr', 'chandauli', 'chitrakoot', 'deoria', 'etah', 'etawah', 'farrukhabad', 'fatehpur', 'firozabad', 'gautam buddha nagar', 'ghaziabad', 'ghazipur', 'gonda', 'gorakhpur', 'hamirpur', 'hapur', 'hardoi', 'hathras', 'jalaun', 'jaunpur', 'jhansi', 'kannauj', 'kanpur dehat', 'kanpur nagar', 'kasganj', 'kaushambi', 'kheri', 'kushi nagar', 'lalitpur', 'lucknow', 'maharajganj', 'mahoba', 'mainpuri', 'mathura', 'mau', 'meerut', 'mirzapur', 'moradabad', 'muzaffarnagar', 'pilibhit', 'pratapgarh', 'prayagraj', 'rae bareli', 'rampur', 'saharanpur', 'sambhal', 'sant kabeer nagar', 'shahjahanpur', 'shamli', 'shravasti', 'siddharth nagar', 'sitapur', 'sonbhadra', 'sultanpur', 'unnao', 'varanasi']

[karnataka] pincode districts:
['bagalkot', 'ballari

In [10]:
manual_map = {
    # Fix incorrect fuzzy mappings
    'faizabad': 'ayodhya',
    # Uttar Pradesh
    'allahabad': 'prayagraj',
    'jyotiba phule nagar': 'amroha',
    'kanshiram nagar': 'kasganj',
    'mahamaya nagar': 'hathras',
    'sant ravidas nagar (bhadohi)': 'bhadohi',
    # Karnataka
    'bangalore': 'bengaluru urban',
    'belgaum': 'belagavi',
    'bellary': 'ballari',
    'gulbarga': 'kalaburagi',
    'mysore': 'mysuru',
    'bijapur': 'vijayapura',
    # Haryana
    'gurgaon': 'gurugram',
    'mewat': 'nuh',
    # West Bengal
    'haora': 'howrah',
    'hugli': 'hooghly',
    'koch bihar': 'coochbehar',
    'dakshin dinajpur': 'dinajpur dakshin',
    'uttar dinajpur': 'dinajpur uttar',
    'north twenty four pargana': '24 paraganas north',
    'south twenty four pargana': '24 paraganas south',
    'paschim medinipur': 'medinipur west',
    'purba medinipur': 'medinipur east',
    # Jharkhand
    'pashchimi singhbhum': 'west singhbhum',
    'purbi singhbhum': 'east singhbum',
    # Uttarakhand
    'garhwal': 'pauri garhwal',
    # Punjab
    'muktsar': 'sri muktsar sahib',
    'sahibzada ajit singh nagar': 's.a.s nagar',
    # Maharashtra
    'bid': 'beed',
    'raigarh': 'raigad',
    # Chhattisgarh
    'koriya': 'korea',
    'uttar bastar kanker': 'kanker',
    # Assam
    'kamrup metropolitan': 'kamrup metro',
    # Gujarat
    'the dangs': 'dang',
    # Andhra Pradesh
    'sri potti sriramulu nello': 'spsr nellore',
    # Telangana
    'warangal rural': 'warangal',
    'warangal urban': 'warangal',
    # Tamil Nadu
    'thoothukkudi': 'tuticorin',
    # Odisha
    'subarnapur': 'sonepur',
    # Jammu and Kashmir
    'punch': 'poonch',
    # Madhya Pradesh
    'khandwa (east nimar)': 'east nimar',
    'khargone (west nimar)': 'khargone',
    # Puducherry
    'puducherry': 'pondicherry',
}

combined_map = {**fuzzy_map, **manual_map}
print(f"Fuzzy: {len(fuzzy_map)}, Manual: {len(manual_map)}, Total: {len(combined_map)}")

Fuzzy: 60, Manual: 43, Total: 102


In [11]:
# Apply combined_map (fuzzy + manual)
nfhs['district_clean'] = nfhs['district_clean'].replace(combined_map)

# Check final match rate
pin_districts = set(pincode['district_clean'].dropna().unique())
nfhs_districts_final = set(nfhs['district_clean'].dropna().unique())
final_match = len(pin_districts & nfhs_districts_final)
still_unmatched = sorted(nfhs_districts_final - pin_districts)

print(f"Final match: {final_match} / {len(nfhs_districts_final)} ({final_match/len(nfhs_districts_final)*100:.1f}%)")
print(f"Still unmatched: {len(still_unmatched)}")
if still_unmatched:
    print(still_unmatched)

Final match: 696 / 697 (99.9%)
Still unmatched: 1
['lakshadweep']


In [12]:
## Step 8. 3-way join → save v4.csv

# Extract one representative (state_clean, district_clean) per pincode_clean
pin_map = (
    pincode[['pincode_clean', 'state_clean', 'district_clean']]
    .drop_duplicates(subset=['pincode_clean', 'state_clean', 'district_clean'])
    .drop_duplicates(subset=['pincode_clean'])  # one row per pincode
)

# 1) facilities ← pincode (on pincode_clean)
fac_pin = facilities.merge(pin_map, on='pincode_clean', how='left')
print(f"facilities + pincode: {fac_pin.shape}")

# 2) + nfhs (on state_clean + district_clean)
nfhs_drop = ['state_ut', 'district_name', 'state_clean', 'district_clean']
nfhs_merge = nfhs.drop(columns=[c for c in nfhs_drop if c in nfhs.columns], errors='ignore')
nfhs_merge.insert(0, 'state_clean', nfhs['state_clean'])
nfhs_merge.insert(1, 'district_clean', nfhs['district_clean'])

v4 = fac_pin.merge(
    nfhs_merge.drop_duplicates(subset=['state_clean', 'district_clean']),
    on=['state_clean', 'district_clean'],
    how='left'
)
print(f"Final join: {v4.shape}")

nfhs_matched = v4['district_clean'].notna() & v4[nfhs.columns.difference(['state_ut','district_name','state_clean','district_clean'])[0]].notna()
print(f"nfhs-matched rows: {nfhs_matched.sum()} / {len(v4)} ({nfhs_matched.sum()/len(v4)*100:.1f}%)")

v4.to_csv("/Users/othree/AIsummit/data/v4.csv", index=False)
print("Saved: /Users/othree/AIsummit/data/v4.csv")

facilities + pincode: (10088, 54)
Final join: (10088, 162)
nfhs-matched rows: 9499 / 10088 (94.2%)
Saved: /Users/othree/AIsummit/data/v4.csv


In [13]:
v4 = pd.read_csv("/Users/othree/AIsummit/data/v4.csv")

/var/folders/zb/yqyg0k1554l0vj7zf6mzbm240000gn/T/ipykernel_34846/3594562566.py:1: DtypeWarning: Columns (28,29,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  v4 = pd.read_csv("/Users/othree/AIsummit/data/v4.csv")


In [14]:
## Step 9. Remove duplicate evidence fields (capability / procedure / equipment)

import ast

def dedup_list_field(val):
    """Remove duplicate items from JSON list string (preserve order, case-insensitive)"""
    if pd.isna(val):
        return val
    try:
        items = ast.literal_eval(val)
    except Exception:
        return val
    if not isinstance(items, list):
        return val

    seen = set()
    result = []
    for item in items:
        key = item.strip().lower()
        if key not in seen:
            seen.add(key)
            result.append(item)
    return str(result)

list_cols = ['capability', 'procedure', 'equipment']

before_counts = {}
after_counts = {}

for col in list_cols:
    before = 0
    after = 0
    for val in v4[col].dropna():
        try:
            items = ast.literal_eval(val)
            if isinstance(items, list):
                before += len(items)
        except:
            pass

    v4[col] = v4[col].apply(dedup_list_field)

    for val in v4[col].dropna():
        try:
            items = ast.literal_eval(val)
            if isinstance(items, list):
                after += len(items)
        except:
            pass

    before_counts[col] = before
    after_counts[col] = after
    removed = before - after
    print(f"{col}: {before:,} items → {after:,} items (removed: {removed:,}, {removed/before*100:.1f}%)")

capability: 232,564 items → 223,701 items (removed: 8,863, 3.8%)
procedure: 167,385 items → 156,033 items (removed: 11,352, 6.8%)
equipment: 60,060 items → 54,609 items (removed: 5,451, 9.1%)


In [15]:
## Step 10. Remove noise from evidence fields

import re

META_PATTERNS = [
    r'^located in\b',
    r'\bis listed (in|as|under|on|for)\b',
    r'^listed as\b',
    r'\blisted under\b',
    r'\b(website|web ?page|webpage)\b',
    r'\b(according to|as per)\b',
]

def is_meta(s):
    sl = s.lower()
    return any(re.search(p, sl) for p in META_PATTERNS)

def count_items(series):
    total = 0
    for val in series.dropna():
        try:
            items = ast.literal_eval(val)
            if isinstance(items, list):
                total += len(items)
        except:
            pass
    return total

def clean_noise(val):
    if pd.isna(val):
        return val
    try:
        items = ast.literal_eval(val)
    except Exception:
        return val
    if not isinstance(items, list):
        return val

    result = []
    for item in items:
        s = str(item).strip()
        if not s:           # 1. empty string
            continue
        if len(s) <= 5:     # 3. too short
            continue
        if len(s) > 500:    # 3. too long
            continue
        if is_meta(s):      # 2. meta sentence
            continue
        result.append(s)

    return str(result) if result else None

for col in ['capability', 'procedure', 'equipment']:
    before_items = count_items(v4[col])
    before_nulls = v4[col].isna().sum()

    v4[col] = v4[col].apply(clean_noise)

    after_items = count_items(v4[col])
    after_nulls = v4[col].isna().sum()

    removed = before_items - after_items
    print(f"{col}: {before_items:,} → {after_items:,} items (removed {removed:,}, {removed/before_items*100:.1f}%), null +{after_nulls - before_nulls}")

capability: 223,701 → 202,613 items (removed 21,088, 9.4%), null +46
procedure: 156,033 → 152,437 items (removed 3,596, 2.3%), null +1110
equipment: 54,609 → 51,883 items (removed 2,726, 5.0%), null +2755


In [16]:
## Step 13. Clean outliers in facilities columns

import ast, re

# 1. Deduplicate specialties
def dedup_specialties(val):
    if pd.isna(val):
        return val
    try:
        items = ast.literal_eval(val)
    except:
        return val
    if not isinstance(items, list):
        return val
    seen = set()
    result = []
    for item in items:
        key = str(item).strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(item)
    return str(result) if result else None

before = 0
for val in v4['specialties'].dropna():
    try:
        items = ast.literal_eval(val)
        if isinstance(items, list):
            before += len(items)
    except: pass

v4['specialties'] = v4['specialties'].apply(dedup_specialties)

after = 0
for val in v4['specialties'].dropna():
    try:
        items = ast.literal_eval(val)
        if isinstance(items, list):
            after += len(items)
    except: pass
print(f"1. specialties dedup: {before:,} → {after:,} items (removed {before-after:,})")

# 2. recency_of_page_update: future dates → None
dates = pd.to_datetime(v4['recency_of_page_update'], errors='coerce')
future_mask = dates > pd.Timestamp('today')
v4.loc[future_mask, 'recency_of_page_update'] = None
print(f"2. recency_of_page_update future dates: {future_mask.sum()} → None")

# 3. address_country: non-'India' values → None
non_india = v4['address_country'] != 'India'
v4.loc[non_india, 'address_country'] = None
print(f"3. address_country invalid values: {non_india.sum()} → None")

# 4. address_stateOrRegion: too long or numeric/JSON → None
def clean_state_region(val):
    if pd.isna(val):
        return val
    s = str(val).strip()
    if len(s) > 60:
        return None
    if re.match(r'^\d+$', s):
        return None
    if s[0] in ('[', '{', '"'):
        return None
    return val

before_nn = v4['address_stateOrRegion'].notna().sum()
v4['address_stateOrRegion'] = v4['address_stateOrRegion'].apply(clean_state_region)
after_nn = v4['address_stateOrRegion'].notna().sum()
print(f"4. address_stateOrRegion outliers: {before_nn - after_nn} → None (unique {v4['address_stateOrRegion'].nunique()})")

# 5. pincode_clean: remove float .0 → 6-digit string
v4['pincode_clean'] = (
    v4['pincode_clean']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
)
v4.loc[~v4['pincode_clean'].str.match(r'^\d{6}$'), 'pincode_clean'] = None
print(f"5. pincode_clean float→string: {v4['pincode_clean'].notna().sum()} valid")

# 6. Drop content_table_id (duplicate of source_content_id)
v4 = v4.drop(columns=['content_table_id'])
print(f"6. content_table_id dropped")

# 7. Drop coordinates (duplicate of latitude/longitude)
v4 = v4.drop(columns=['coordinates'])
print(f"7. coordinates dropped")

print(f"\nFinal: {v4.shape}")
v4.to_csv("/Users/othree/AIsummit/data/v4.csv", index=False)
print("Saved")

1. specialties dedup: 279,003 → 118,221 items (removed 160,782)
2. recency_of_page_update future dates: 1 → None
3. address_country invalid values: 88 → None
4. address_stateOrRegion outliers: 19 → None (unique 235)
5. pincode_clean float→string: 9920 valid
6. content_table_id dropped
7. coordinates dropped

Final: (10088, 160)


Saved


In [17]:
## Step 14. Additional cleaning

import ast, re

# ── Common dedup function ──────────────────────────────────────────
def dedup_list_str(val):
    if pd.isna(val):
        return val
    try:
        items = ast.literal_eval(val)
    except:
        return val
    if not isinstance(items, list):
        return val
    seen = set()
    result = []
    for item in items:
        key = str(item).strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(item)
    return str(result) if result else None

def count_list_col(series):
    total = 0
    for v in series.dropna():
        try:
            items = ast.literal_eval(v)
            if isinstance(items, list):
                total += len(items)
        except:
            pass
    return total

# 1. source_types dedup
v4['source_types'] = v4['source_types'].apply(dedup_list_str)
print("1. source_types dedup done")

# 2. affiliationTypeIds dedup
v4['affiliationTypeIds'] = v4['affiliationTypeIds'].apply(dedup_list_str)
print("2. affiliationTypeIds dedup done")

# 3. specialties dedup
v4['specialties'] = v4['specialties'].apply(dedup_list_str)
print("3. specialties dedup done")

# 4. Remove address from name
ADDR_KW = re.compile(
    r'\b(road|street|nagar|colony|sector|lane|plot|floor|block|phase|'
    r'main|cross|layout|bypass|no\.|hsr|mig|lig|india|maharashtra|kerala|'
    r'karnataka|gujarat|rajasthan|punjab|haryana|uttar pradesh|tamil nadu)\b',
    re.IGNORECASE
)
MEDICAL_KW = re.compile(
    r'\b(hospital|clinic|centre|center|medical|health|care|nursing|'
    r'institute|college|foundation|trust|society)\b',
    re.IGNORECASE
)

def clean_name(val):
    if pd.isna(val):
        return val
    s = str(val).strip()

    # Pattern 1: "Name - Address" (number or address keyword after dash)
    dash_parts = s.split(' - ', 1)
    if len(dash_parts) == 2 and (re.search(r'\d', dash_parts[1]) or ADDR_KW.search(dash_parts[1])):
        s = dash_parts[0].strip()

    # Pattern 2: "Name, City, State[, Country]" — remove trailing parts if short and no medical keyword
    parts = [p.strip() for p in s.split(',')]
    if len(parts) >= 3:
        while len(parts) > 1:
            last = parts[-1]
            if len(last) <= 25 and not MEDICAL_KW.search(last):
                parts.pop()
            else:
                break
        s = ', '.join(parts)

    return s.strip() or None

before_sample = v4['name'].dropna().head(3).tolist()
v4['name'] = v4['name'].apply(clean_name)
print(f"4. name address removal done (e.g. {before_sample[0]!r} → {v4['name'].iloc[0]!r})")

# 5. Strip spaces from address_zipOrPostcode + validate 6-digit format
v4['address_zipOrPostcode'] = (
    v4['address_zipOrPostcode']
    .astype(str)
    .str.replace(' ', '', regex=False)
    .str.strip()
)
v4.loc[~v4['address_zipOrPostcode'].str.match(r'^\d{6}$'), 'address_zipOrPostcode'] = None
print(f"5. address_zipOrPostcode cleaned: {v4['address_zipOrPostcode'].notna().sum()} valid")

# 6. source_urls dedup
before = count_list_col(v4['source_urls'])
v4['source_urls'] = v4['source_urls'].apply(dedup_list_str)
after = count_list_col(v4['source_urls'])
print(f"6. source_urls dedup: {before:,} → {after:,} URLs (removed {before - after:,})")

# 7. Drop unnecessary columns
drop_cols = [c for c in ['source_ids', 'phone_numbers', 'websites', 'address_country', 'address_countryCode', 'countries'] if c in v4.columns]
v4 = v4.drop(columns=drop_cols)
print(f"7. Columns dropped: {drop_cols}")

print(f"\nFinal: {v4.shape}")
v4.to_csv("/Users/othree/AIsummit/data/v4.csv", index=False)
print("Saved")

1. source_types dedup done
2. affiliationTypeIds dedup done
3. specialties dedup done
4. name address removal done (e.g. 'Aravind Eye Hospital' → 'Aravind Eye Hospital')
5. address_zipOrPostcode cleaned: 9926 valid
6. source_urls dedup: 60,000 → 57,384 URLs (removed 2,616)
7. Columns dropped: ['source_ids', 'phone_numbers', 'websites', 'address_country', 'address_countryCode', 'countries']

Final: (10088, 154)
Saved


In [18]:
# ## Step 12. Drop unnecessary columns

drop_cols = [
    'acceptsVolunteers',
    'area',
    'affiliationTypeIds',
    'number_of_facts_about_the_organization',
    'post_metrics_post_count',
    'post_metrics_most_recent_social_media_post_date',
    'engagement_metrics_n_engagements',
]

v4 = v4.drop(columns=drop_cols)
print(f"Removed: {len(drop_cols)} columns")
print(f"Remaining columns: {v4.shape[1]}")

v4.to_csv("/Users/othree/AIsummit/data/v4.csv", index=False)
print("Saved")

Removed: 7 columns
Remaining columns: 147
Saved


In [19]:
## Step 11. Drop rows with no usable data

location = v4['latitude'].notna() | v4['address_city'].notna() | v4['address_stateOrRegion'].notna()
evidence = v4[['description','capability','procedure','equipment','specialties']].notna().any(axis=1)
name = v4['name'].notna()

before = len(v4)
v4 = v4[location & evidence & name].reset_index(drop=True)
after = len(v4)

print(f"{before} → {after} rows (removed: {before - after})")

v4.to_csv("/Users/othree/AIsummit/data/v4.csv", index=False)
print("Saved")

10088 → 10007 rows (removed: 81)
Saved


In [20]:
v4.to_csv("/Users/othree/AIsummit/data/v4.csv", index=False)
print(f"Saved: {v4.shape}")

# Missing values for all v4 columns
missing = v4.isnull().sum()
pct = (missing / len(v4) * 100).round(1)
summary = pd.DataFrame({'missing': missing, 'missing_%': pct})
print()
print(summary.to_string())

Saved: (10007, 147)

                                                                 missing  missing_%
unique_id                                                              0        0.0
source_types                                                          28        0.3
source_content_id                                                      0        0.0
name                                                                   0        0.0
organization_type                                                      1        0.0
officialPhone                                                        499        5.0
email                                                               1456       14.5
officialWebsite                                                     1607       16.1
yearEstablished                                                     5219       52.2
facebookLink                                                         118        1.2
address_line1                                          